# Convergence and the damping factor

## Purpose

This is an **extension** (research question 6), the second to be cut if
time runs short. It answers two questions on the real CAIDA AS graph:
how the damping factor `alpha` drives the convergence of the power
method and the ranking it produces; and whether a second solver,
Gauss-Seidel on the linear form of PageRank, does better.

The theory in three sentences. The power method contracts by at most
`alpha` per step, so `log(tol) / log(alpha)` is an *upper bound* on the
iterations to a fixed tolerance; the true rate is `abs(lambda_2(G))`, the
subdominant eigenvalue of the Google matrix, and a standard result
(Haveliwala and Kamvar, *The second eigenvalue of the Google matrix*,
Stanford 2003) gives `lambda_2(G) = alpha * lambda_2(P)`, so the observed
`rate / alpha` estimates `abs(lambda_2(P))` and the bound is loose
whenever that is well below 1. Gauss-Seidel solves the same linear system
`(I - alpha P^T) x = (1 - alpha) v`, and its asymptotic rate is the
spectral radius of *its own* iteration matrix `M^{-1} U` (with `A = I -
alpha P^T = M - U`, `M = tril(A)`), which is not bounded by `alpha` and
depends on the order in which the nodes are swept. Larger `alpha` weakens
teleportation and lets the ranking follow the link structure more closely.

So this notebook does **not** claim Gauss-Seidel is faster. Whether it
wins is a property of the graph and of the node ordering, not of the
method; here the nodes are swept in ascending-ASN (index) order, and the
notebook measures what that ordering gives on this graph. Node orderings
are out of scope (the "extension of an extension").

The solvers live in `pagerank`; this notebook loads, runs, tabulates and
plots.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve_triangular

from pagerank import (
    from_edge_list,
    pagerank_power,
    pagerank_gauss_seidel,
    ranking,
    top_k_overlap,
    kendall_tau,
)
from pagerank.caida import load_build

REPO = Path.cwd().parent
PROCESSED = REPO / 'data' / 'processed'
FIGURES = REPO / 'results' / 'figures'
TABLES = REPO / 'results' / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

SNAPSHOT = '20260901'
PRIMARY_NPZ = PROCESSED / f'{SNAPSHOT}_peering-both_vote-c2p_ixp-kept.npz'
NONE_NPZ = PROCESSED / f'{SNAPSHOT}_peering-none_vote-c2p_ixp-kept.npz'

ALPHAS = [0.50, 0.70, 0.85, 0.90, 0.95, 0.99]
TOL = 1e-10
MAX_ITER = 5000
t_start = time.perf_counter()

## 1. Setup: the two graphs and one Gauss-Seidel sweep

Both solvers are exercised on two builds from notebook 01. The **primary**
graph (peering `both`, vote customer-to-provider, IXPs kept) has 80,510
nodes and no dangling node: with peering every node keeps an out-edge.
The **peering-`none`** graph drops the peer-to-peer edges, leaving only
customer-to-provider links; its 494 provider-free ASes then dangle, so
the dangling handling of both solvers is tested on real data and not only
on the toy fixtures.

The two builds have very different structure. The primary graph, with
~81% of its edges from symmetric peering, is densely connected; the
peering-`none` graph is the customer-to-provider hierarchy alone, which
is close to acyclic and flows upward towards the core. That difference
is what makes Gauss-Seidel behave oppositely on the two, as section 3
shows.

One Gauss-Seidel sweep on the primary graph is timed and printed: it is
a single sparse lower-triangular solve on an 80,510-node system.

In [ ]:
primary_build = load_build(PRIMARY_NPZ)
none_build = load_build(NONE_NPZ)
assert np.array_equal(primary_build.asns, none_build.asns)  # same node indexing
asns = primary_build.asns

primary = from_edge_list(primary_build.edges, primary_build.n)
none = from_edge_list(none_build.edges, none_build.n)

print(f'primary: nodes {primary.n}, edges {primary.n_edges}, '
      f'dangling {int(primary.dangling.sum())}')
print(f'peering-none: nodes {none.n}, edges {none.n_edges}, '
      f'dangling {int(none.dangling.sum())}')
assert primary.n == 80510 and int(primary.dangling.sum()) == 0
assert int(none.dangling.sum()) == 494

# Time one Gauss-Seidel sweep on the primary graph: build M = tril(A) and
# U once, then a single lower-triangular solve (exactly what one sweep of
# pagerank_gauss_seidel does internally).
n = primary.n
pt = primary.transpose
v = np.full(n, 1.0 / n)
a = (sp.identity(n, dtype=np.float64, format='csr') - 0.85 * pt).tocsr()
m_mat = sp.tril(a, k=0, format='csr')
upper = (0.85 * sp.triu(pt, k=1)).tocsr()
t0 = time.perf_counter()
_ = spsolve_triangular(m_mat, (1 - 0.85) * v + upper @ v, lower=True)
sweep_ms = (time.perf_counter() - t0) * 1000
print(f'one Gauss-Seidel sweep on the primary graph: {sweep_ms:.1f} ms')

## 2. Why the two methods converge at the rates they do

Each method's asymptotic convergence rate is the spectral radius of its
own iteration matrix. For the power method that is `abs(lambda_2)`, the
subdominant eigenvalue of the (dangling-repaired) Google matrix `G =
alpha P + (1 - alpha) 1 v^T`. For Gauss-Seidel it is `rho(M^{-1} U)`,
with `A = I - alpha P^T`, `M = tril(A)`, `U = -triu(A, 1)`. Which of the
two is smaller is a property of the graph and of the sweep ordering, not
of the method.

Computed exactly on the two 4×4 fixtures (dense, so the eigenvalues are
the real ones, not estimates):

- **Toy graph, index order.** Gauss-Seidel's rate exceeds the power
  method's, so it is genuinely *slower* here — the ordering does not suit
  the link structure. This is the reversal of the "typically fewer
  sweeps" intuition that holds for many web graphs.
- **Dangling fixture, index order.** The graph is acyclic and its node
  indices are already a topological order, so the lower-triangular solve
  resolves every unknown in one forward pass: Gauss-Seidel's rate is 0
  and it is exact after a single sweep.

Both facts are pinned by the test suite
(`tests/test_gauss_seidel.py`). They are printed again here so the
notebook's framing rests on numbers in front of the reader.

In [ ]:
import sys
sys.path.insert(0, str(REPO))  # reach tests/ for the shared fixtures
from tests.test_toy_graph import TOY_EDGES, TOY_N, DANGLING_EDGES, DANGLING_N


def spectral_rates(edges, n, alpha=0.85):
    """Return (abs(lambda_2) of the repaired Google matrix, rho(M^-1 U))."""
    g = from_edge_list(edges, n)
    pt_dense = g.transpose.toarray()
    v = np.full(n, 1.0 / n)
    # Repaired P: dangling rows (zero rows of P) are replaced by v, which
    # is exactly what the power method's dangling term does.
    p = pt_dense.T.copy()
    p[g.dangling, :] = v
    google = alpha * p + (1 - alpha) * np.outer(np.ones(n), v)
    lam2 = np.sort(np.abs(np.linalg.eigvals(google)))[::-1][1]
    a_mat = np.eye(n) - alpha * pt_dense
    m_lower = np.tril(a_mat)
    u_upper = -np.triu(a_mat, 1)
    rho_gs = np.max(np.abs(np.linalg.eigvals(np.linalg.solve(m_lower, u_upper))))
    return float(lam2), float(rho_gs)


toy_lam2, toy_rho = spectral_rates(TOY_EDGES, TOY_N)
dang_lam2, dang_rho = spectral_rates(DANGLING_EDGES, DANGLING_N)
print(f'toy graph      : power abs(lambda_2) = {toy_lam2:.3f}   '
      f'Gauss-Seidel rho(M^-1 U) = {toy_rho:.3f}  (GS slower)')
print(f'dangling fixture: power abs(lambda_2) = {dang_lam2:.3f}   '
      f'Gauss-Seidel rho(M^-1 U) = {dang_rho:.3f}  (GS exact in one sweep)')
assert toy_rho > toy_lam2
assert dang_rho < 1e-12

## 3. Iterations, time and observed rate versus alpha

For each graph, each `alpha`, each solver: the iteration count to reach
`tol = 1e-10`, whether it converged, the observed asymptotic rate (the
median of `r[k+1] / r[k]` over the second half of the residual history),
and the max absolute difference between the two solvers' vectors (which
must be below `1e-9`, since they solve the same system). Two derived
columns are added: `theory_power_iterations = ceil(log(tol)/log(alpha))`,
the upper bound, and `rate_over_alpha = power_observed_rate / alpha`, the
estimate of `abs(lambda_2(P))`. These reproducible columns go to
`03_convergence_vs_alpha.csv`.

The wall-time measurements — total seconds and milliseconds per iteration
for each solver — vary from run to run, so they are written to a separate
`03_timings.csv` to keep the main table reproducible byte for byte. They
are still printed. The per-iteration time is the point the design asked
to be stated plainly: a Gauss-Seidel sweep is a triangular solve and
costs about ten times a sparse mat-vec, so even where the two methods
take a similar number of iterations the power method is far cheaper per
step.

In [ ]:
def observed_rate(residuals):
    """Median successive-residual ratio over the tail (second half)."""
    r = np.asarray(residuals, dtype=np.float64)
    r = r[r > 0]
    if r.size < 4:
        return float('nan')
    tail = r[r.size // 2:]
    ratios = tail[1:] / tail[:-1]
    return float(np.median(ratios))


conv_rows = []
timing_rows = []
for graph_name, g in (('primary', primary), ('peering_none', none)):
    for alpha in ALPHAS:
        t0 = time.perf_counter()
        rp = pagerank_power(g, alpha=alpha, tol=TOL, max_iter=MAX_ITER)
        t_power = time.perf_counter() - t0
        t0 = time.perf_counter()
        rg = pagerank_gauss_seidel(g, alpha=alpha, tol=TOL, max_iter=MAX_ITER)
        t_gs = time.perf_counter() - t0
        max_diff = float(np.abs(rp.vector - rg.vector).max())
        assert max_diff < 1e-9, (graph_name, alpha, max_diff)
        power_rate = observed_rate(rp.residuals)
        # Reproducible columns only: iteration counts, convergence flags,
        # observed rates, the loose upper bound and rate/alpha (the estimate
        # of abs(lambda_2(P))). No wall times here.
        conv_rows.append({
            'graph': graph_name,
            'alpha': alpha,
            'power_iterations': rp.iterations,
            'theory_power_iterations': int(np.ceil(np.log(TOL) / np.log(alpha))),
            'power_converged': rp.converged,
            'power_observed_rate': power_rate,
            'rate_over_alpha': power_rate / alpha,
            'gs_iterations': rg.iterations,
            'gs_converged': rg.converged,
            'gs_observed_rate': observed_rate(rg.residuals),
            'max_abs_diff': max_diff,
        })
        timing_rows.append({
            'graph': graph_name,
            'alpha': alpha,
            'power_seconds': t_power,
            'power_ms_per_iter': 1000 * t_power / rp.iterations,
            'gs_seconds': t_gs,
            'gs_ms_per_iter': 1000 * t_gs / rg.iterations,
        })

conv = pd.DataFrame(conv_rows)
conv_path = TABLES / '03_convergence_vs_alpha.csv'
conv.to_csv(conv_path, index=False)
print('wrote', conv_path.relative_to(REPO))

timings = pd.DataFrame(timing_rows)
timings_path = TABLES / '03_timings.csv'
timings.to_csv(timings_path, index=False)
print('wrote', timings_path.relative_to(REPO))

# Per-step cost ratio, averaged over alpha on the primary graph.
prim_t = timings[timings['graph'] == 'primary']
step_ratio = (prim_t['gs_ms_per_iter'] / prim_t['power_ms_per_iter']).mean()
print(f'Gauss-Seidel sweep vs power mat-vec, primary graph: '
      f'{step_ratio:.0f}x more expensive per step')
conv.round(4)

The plot: iterations against `alpha` on a log y-axis, both solvers, both
graphs, with the curve `log(tol) / log(alpha)` drawn as the *upper
bound* it is. The measured power-method counts sit well below that bound
and the gap widens with `alpha` — tenfold at 0.99 on the primary graph —
because the true rate is `alpha * abs(lambda_2(P))`, not `alpha`. On the
**peering-`none`** graph Gauss-Seidel sits well below the power method at
every `alpha`.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
alphas = np.array(ALPHAS)
theory = np.log(TOL) / np.log(alphas)
styles = {
    ('primary', 'power'): ('o-', 'tab:blue'),
    ('primary', 'gs'): ('s-', 'tab:red'),
    ('peering_none', 'power'): ('o--', 'tab:cyan'),
    ('peering_none', 'gs'): ('s--', 'tab:orange'),
}
for graph_name in ('primary', 'peering_none'):
    sub = conv[conv['graph'] == graph_name].sort_values('alpha')
    fmt, col = styles[(graph_name, 'power')]
    ax.semilogy(sub['alpha'], sub['power_iterations'], fmt, color=col,
                label=f'power ({graph_name})')
    fmt, col = styles[(graph_name, 'gs')]
    ax.semilogy(sub['alpha'], sub['gs_iterations'], fmt, color=col,
                label=f'Gauss-Seidel ({graph_name})')
ax.semilogy(alphas, theory, 'k:', label='upper bound log(tol)/log(alpha)')
ax.set_xlabel('alpha')
ax.set_ylabel('iterations to tol 1e-10 (log)')
ax.set_title('Iterations versus alpha')
ax.legend(fontsize=8)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
iter_path = FIGURES / '03_iterations_vs_alpha.png'
fig.savefig(iter_path, dpi=150)
print('wrote', iter_path.relative_to(REPO))
plt.show()

## 4. Residual curves at alpha 0.85

The L1 residual against iteration on a semilog axis for both solvers on
the primary graph at `alpha = 0.85`, with the geometric bound `alpha**k`
scaled to the first residual. Both curves are straight lines on this
axis (geometric convergence), with slopes close to the observed rates in
the table — at 0.85 the two are near enough that the iteration counts are
almost equal.

In [ ]:
rp85 = pagerank_power(primary, alpha=0.85, tol=TOL, max_iter=MAX_ITER)
rg85 = pagerank_gauss_seidel(primary, alpha=0.85, tol=TOL, max_iter=MAX_ITER)

fig, ax = plt.subplots(figsize=(7.5, 5))
kp = np.arange(1, len(rp85.residuals) + 1)
kg = np.arange(1, len(rg85.residuals) + 1)
ax.semilogy(kp, rp85.residuals, 'o-', color='tab:blue', markersize=3,
            label='power method')
ax.semilogy(kg, rg85.residuals, 's-', color='tab:red', markersize=3,
            label='Gauss-Seidel')
bound = rp85.residuals[0] * 0.85 ** (kp - 1)
ax.semilogy(kp, bound, 'k:', label='alpha**k bound (scaled)')
ax.set_xlabel('iteration k')
ax.set_ylabel('L1 residual (log)')
ax.set_title('Residual decay at alpha = 0.85 (primary graph)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
res_path = FIGURES / '03_residuals_alpha085.png'
fig.savefig(res_path, dpi=150)
print('wrote', res_path.relative_to(REPO))
plt.show()

## 5. The observed power-method rate against alpha

The bound `log(tol)/log(alpha)` treats the rate as `alpha`. The measured
rate is `alpha * abs(lambda_2(P))` (Haveliwala and Kamvar, 2003), so
`rate_over_alpha = power_observed_rate / alpha` estimates
`abs(lambda_2(P))`, the subdominant eigenvalue of the transition matrix
itself. That number is what the bound ignores, and it is what makes the
bound loose. It is plotted against `alpha` for both graphs below.

The two graphs behave differently, and both are reported as measured:

- **peering-`none`.** `rate_over_alpha` is about **0.53 and flat** across
  every `alpha`, exactly the constant the theory predicts when
  `abs(lambda_2(P))` does not depend on `alpha` (it is a property of `P`
  alone). The observed rate is therefore close to `0.53 * alpha`, well
  under the bound.
- **primary.** `rate_over_alpha` is not flat: it **drifts from about 0.84
  at `alpha = 0.5` to about 0.94 at `alpha = 0.99`.** This is reported,
  not explained away: the tail estimate may not be fully asymptotic at
  high `alpha`, or several eigenvalues of `P` are close in modulus so the
  single-ratio estimate mixes them. Either way the rate stays below
  `alpha`, so the bound holds but is loose — tenfold in iterations at
  `alpha = 0.99` (215 against the bound's 2,292).

In [ ]:
prim = conv[conv['graph'] == 'primary'].sort_values('alpha')
none_c = conv[conv['graph'] == 'peering_none'].sort_values('alpha')

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(prim['alpha'], prim['rate_over_alpha'], 'o-', color='tab:blue',
        label='primary')
ax.plot(none_c['alpha'], none_c['rate_over_alpha'], 's--', color='tab:orange',
        label='peering none')
ax.set_xlabel('alpha')
ax.set_ylabel('rate / alpha  (estimate of abs(lambda_2(P)))')
ax.set_title('Observed rate divided by alpha versus alpha')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Reported as measured: flat near 0.53 on peering none, drifting on primary.
print('rate / alpha by alpha:')
print(f"  primary     : {[round(x, 2) for x in prim['rate_over_alpha']]}")
print(f"  peering none: {[round(x, 2) for x in none_c['rate_over_alpha']]}")
# The bound holds (rate below alpha) on both graphs; it is simply loose.
assert (conv['power_observed_rate'] < conv['alpha']).all()

## 6. Ranking stability across alpha

Does the choice of `alpha = 0.85` in notebook 02 change the conclusions?
For each `alpha`, the PageRank vector on the primary graph is compared
against the `alpha = 0.85` vector by two measures: Kendall's tau-b over
the union of the two top-1000 sets, and the top-10 overlap. Results go to
`03_ranking_vs_alpha.csv`.

To see the direction of the churn rather than just its size, a second
table records, for every AS that reaches the top 10 at `alpha` 0.50, 0.85
or 0.99, its rank at those three values together with its customer and
peer counts from the snapshot. It is sorted by rank at 0.85 and saved to
`03_rank_trajectories.csv`.

In [ ]:
pr_by_alpha = {
    alpha: pagerank_power(primary, alpha=alpha, tol=TOL, max_iter=MAX_ITER).vector
    for alpha in ALPHAS
}
ref = pr_by_alpha[0.85]
ref_top10 = set(int(i) for i in ranking(ref)[:10])

rank_rows = []
for alpha in ALPHAS:
    vec = pr_by_alpha[alpha]
    union_top = np.union1d(ranking(vec)[:1000], ranking(ref)[:1000])
    rank_rows.append({
        'alpha': alpha,
        'tau_b_top1000_union_vs_085': kendall_tau(vec, ref, subset=union_top),
        'overlap_top10_vs_085': top_k_overlap(vec, ref, 10),
    })
ranking_tbl = pd.DataFrame(rank_rows)
ranking_path = TABLES / '03_ranking_vs_alpha.csv'
ranking_tbl.to_csv(ranking_path, index=False)
print('wrote', ranking_path.relative_to(REPO))

for alpha in (0.50, 0.99):
    top10 = set(int(i) for i in ranking(pr_by_alpha[alpha])[:10])
    entered = sorted(int(asns[i]) for i in top10 - ref_top10)
    left = sorted(int(asns[i]) for i in ref_top10 - top10)
    print(f'alpha={alpha}: entered top 10 vs 0.85 = {entered}; left = {left}')
ranking_tbl.round(4)

In [ ]:
# Rank trajectories of the top-10 ASes across alpha, with their customer
# and peer counts from the raw snapshot. Customers of an AS = number of
# c2p (-1) rows where it is the provider (as1); peers = number of p2p (0)
# rows it appears in on either side. Nothing here is hard-coded: the AS
# set is the union of the top 10 at the three alphas.
from pagerank.caida import read_as_rel

snap = read_as_rel(REPO / 'data' / 'raw' / f'{SNAPSHOT}.as-rel.bz2')
rel = snap.relationships
c2p = rel[rel[:, 2] == -1]
p2p = rel[rel[:, 2] == 0]
n_customers = np.zeros(int(asns.max()) + 1, dtype=np.int64)
np.add.at(n_customers, c2p[:, 0], 1)  # provider gains one customer per c2p row
n_peers = np.zeros(int(asns.max()) + 1, dtype=np.int64)
np.add.at(n_peers, p2p[:, 0], 1)
np.add.at(n_peers, p2p[:, 1], 1)

TRAJ_ALPHAS = [0.50, 0.85, 0.99]
# 1-based rank position of every node at each of the three alphas.
rank_pos = {}
for alpha in TRAJ_ALPHAS:
    order = ranking(pr_by_alpha[alpha])
    pos = np.empty(primary.n, dtype=np.int64)
    pos[order] = np.arange(1, primary.n + 1)
    rank_pos[alpha] = pos

# Union of the top-10 node indices across the three alphas.
union_top10 = sorted(
    set().union(*(set(int(i) for i in ranking(pr_by_alpha[a])[:10])
                  for a in TRAJ_ALPHAS))
)
traj_rows = []
for node in union_top10:
    asn = int(asns[node])
    traj_rows.append({
        'asn': asn,
        'customers': int(n_customers[asn]),
        'peers': int(n_peers[asn]),
        'rank_050': int(rank_pos[0.50][node]),
        'rank_085': int(rank_pos[0.85][node]),
        'rank_099': int(rank_pos[0.99][node]),
    })
traj_tbl = pd.DataFrame(traj_rows).sort_values('rank_085').reset_index(drop=True)
traj_path = TABLES / '03_rank_trajectories.csv'
traj_tbl.to_csv(traj_path, index=False)
print('wrote', traj_path.relative_to(REPO))

ref_top10 = set(int(i) for i in ranking(pr_by_alpha[0.85])[:10])
top10_099 = set(int(i) for i in ranking(pr_by_alpha[0.99])[:10])
left_099 = sorted(int(asns[i]) for i in ref_top10 - top10_099)
entered_099 = sorted(int(asns[i]) for i in top10_099 - ref_top10)
print('leave top 10 at 0.99 vs 0.85:', left_099)
print('enter top 10 at 0.99 vs 0.85:', entered_099)
traj_tbl

The ranking is **not** stable across `alpha`, and the instability has a
direction. Kendall's tau-b over the top-1000 union against `alpha = 0.85`
is 0.55 at `alpha = 0.5` and 0.64 at `alpha = 0.99`, and the top-10
overlap is 0.7 at both ends: three of the ten change at each extreme.

The trajectory table shows what moves. Three ASes leave the top 10 at
`alpha = 0.99` — AS174, AS3356 and AS1000 — and three enter. The pattern
across the three columns is cleaner than the top-10 swap alone: the large
transit providers fall steadily as `alpha` rises (AS174 from rank 1 to 13,
AS3356 from 2 to 16, AS7018 from 7 to 200), while the heavily peered
networks climb (AS39120 from 22 to 5, AS32590 from 16 to 7). AS1000 is the
exception that keeps the reading honest: it has only 3 customers but 5,138
peers, yet it falls from rank 5 to 22, so "peering hubs rise" is a
tendency and not a law — one heavily peered network moves the other way.

The direction is the central finding of notebook 02 (peering hubs collect
score under the peering-`both` model) getting *stronger* as `alpha` grows
and teleportation weakens. The consequence for the project is plain:
`alpha = 0.85` is a modelling choice that has to be stated and justified,
not a neutral default, because the top of the ranking moves with it.

## 7. What this notebook established

- `log(tol)/log(alpha)` is an upper bound on the power method's
  iterations, and on these graphs it is loose: the true rate is
  `alpha * abs(lambda_2(P))`, so iterations grow from a few dozen at
  `alpha = 0.5` to a couple of hundred at `alpha = 0.99` (against the
  bound's 2,292). The estimate `rate / alpha` is flat near 0.53 on the
  peering-`none` graph, as the theory predicts, and drifts from 0.84 to
  0.94 on the primary graph, reported as measured.
- Each method converges at the spectral radius of its own iteration
  matrix. On the toy graph in index order Gauss-Seidel is slower
  (`rho(M^-1 U) = 0.668 > abs(lambda_2) = 0.601`); on the acyclic
  dangling fixture in topological order it is exact after one sweep.
- On the real graph the outcome depends on the build and the sweep
  ordering, not on the method. On the peering-`none` graph (the
  near-acyclic customer-to-provider hierarchy) Gauss-Seidel in ASN order
  needs far fewer iterations at every `alpha`; on the primary graph
  (dominated by symmetric peering) the two are close near `alpha = 0.85`
  and the power method pulls clearly ahead as `alpha` approaches 1.
- Per step a Gauss-Seidel sweep is a triangular solve costing about ten
  times a sparse mat-vec, so on the primary graph the power method is the
  cheaper solver in wall time at every `alpha`, iteration counts aside.
- **Answer to the second question: no, Gauss-Seidel does not beat the
  power method on this graph in this ordering.** Its advantage is a
  property of the node ordering relative to the link structure, not of
  the method; it appears on the near-acyclic peering-`none` build and
  disappears on the peering-dominated primary build. Node orderings are
  out of scope.
- The ranking is alpha-sensitive. Tau-b over the top-1000 union against
  `alpha = 0.85` falls to 0.55 at `alpha = 0.5` and 0.64 at `alpha =
  0.99`. At 0.99 three ASes leave the top 10 — AS174, AS3356 and AS1000 —
  and three enter; across `alpha` the large transit providers fall
  steadily and most peering hubs climb, AS1000 (3 customers, 5,138 peers)
  being the exception that falls. The peering-hub dominance found in
  notebook 02 strengthens as `alpha` grows, so `alpha = 0.85` is a choice
  that must be stated and justified, not a neutral default.

In [ ]:
print(f'total notebook wall time: {time.perf_counter() - t_start:.1f} s')